# Jane Street Real-Time Market Data Forecasting: High-Frequency Ensembles & Stateful Streaming

## I. Introduction & The Real-Time Challenge
In high-frequency trading, anticipating the immediate future price movement of an asset requires processing massive streams of order book data in real-time. In this competition, the objective was to predict responder_6—a highly anonymized target representing a stock's future price movement—across 39 different symbols.

The Metric: Weighted R²
Because financial data is exceptionally noisy, predictions must be confidence-weighted. The competition evaluates models using a Weighted R² metric. A model must not only predict the direction of the asset accurately but also scale its predictions based on the weight column (representing the trading volume or conviction of that specific time step).

The Streaming Constraint (The Engineering Hurdle)
Unlike standard data science competitions where predictions are generated in bulk, this competition utilizes Kaggle's kaggle_evaluation API. Data is fed to the model strictly one time_id at a time. This constraint completely breaks standard time-series models. You cannot simply use pandas .shift(1) to look at the previous time step, because previous time steps are no longer in memory. Building an engine that can track historical state across 327,000+ sequential time steps without exhausting CPU RAM or GPU VRAM became the ultimate challenge of this project.

## II. Feature Engineering & Temporal Dynamics

Extensive Exploratory Data Analysis (EDA) revealed two critical structural properties of the dataset:

1. Autocorrelation Decay: The lag-1 responder is the dominant signal (correlation ~0.89). This signal decays rapidly and is effectively gone by lag-20.

2. Cross-Sectional Alpha: A stock does not move in a vacuum. By evaluating all 39 symbols at a single time step, we can calculate the "market weather."

To capture this, the feature engineering pipeline calculates rolling statistics and cross-symbol relative metrics (e.g., r6_lag_vs_market), allowing the models to instantly understand a symbol's momentum relative to the broader market index.

In [ ]:
import polars as pl
import torch
import torch.nn as nn
import numpy as np

def apply_feature_engineering(lf: pl.LazyFrame) -> pl.LazyFrame:
    """Applies temporal and cross-sectional feature engineering."""
    grp = ["date_id", "symbol_id"]
    cross_grp = ["date_id", "time_id"]
    
    return (
        lf.sort(["date_id", "symbol_id", "time_id"])
        
        # 1. Temporal Dynamics: Rolling Stats & Lags
        .with_columns([
            pl.col(f"responder_{i}").shift(1).over(grp).alias(f"responder_{i}_lag_1")
            for i in range(9)
        ])
        .with_columns([
            pl.col("responder_6_lag_1").rolling_mean(5, min_samples=1).over(grp).alias("r6_lag_roll5_mean"),
            pl.col("responder_6_lag_1").rolling_std(5, min_samples=2).over(grp).alias("r6_lag_roll5_std"),
        ])
        
        # 2. Cross-Sectional Alpha: Market relative features
        .with_columns([
            pl.col("responder_6_lag_1").mean().over(cross_grp).alias("market_r6_lag_mean"),
            pl.col("feature_00").mean().over(cross_grp).alias("market_f00_mean"),
        ])
        .with_columns([
            (pl.col("responder_6_lag_1") - pl.col("market_r6_lag_mean")).alias("r6_lag_vs_market"),
            (pl.col("feature_00") - pl.col("market_f00_mean")).alias("f00_vs_market"),
        ])
        .fill_null(0.0)
    )

## III. Validation Strategy & Market Regime Shifts

Standard randomized K-Fold cross-validation fails in finance due to temporal leakage. I utilized a strict Walk-Forward Cross-Validation strategy, training on Partitions 0-7 and holding out Partitions 8-9 for evaluation.

This strict split exposed a severe market regime shift. Tree-based models (LightGBM) that achieved stellar scores on the training partitions catastrophically failed on the holdout set, dropping to a -0.4479 R². The hard, geometric thresholds of the decision trees over-extrapolated to the new market environment. This failure proved the necessity of building robust, continuous neural architectures that could generalize through structural market changes.

## IV. The Uncorrelated Ensemble: Sequential vs. Cross-Sectional

To build a robust prediction engine, I ensembled two custom PyTorch architectures with fundamentally opposed inductive biases:

1. The GRU (Temporal Focus): Treats a single stock as a sequence over time. It relies on its hidden state to remember the momentum of previous ticks.

2. The Cross-Symbol Transformer (Spatial Focus): Instead of looking backward in time, the CST uses Multi-Head Attention to look "sideways" at all 39 stocks simultaneously within a single millisecond, dynamically weighting the influence of other symbols on the target stock.

By optimizing the blend of 3 different CST seed variations and 1 GRU using a SciPy SLSQP optimizer, the ensemble naturally canceled out individual model noise and achieved an exceptionally stable out-of-sample score.

In [ ]:
class CrossSymbolTransformer(nn.Module):
    """
    Evaluates cross-sectional market dynamics. 
    Instead of processing sequences over time, it applies Multi-Head Attention 
    across all 39 active symbols at a single time_id snapshot.
    """
    def __init__(self, num_features: int, hidden_dim: int = 64, 
                 num_heads: int = 4, num_targets: int = 4, dropout: float = 0.2):
        super().__init__()
        self.input_proj = nn.Linear(num_features, hidden_dim)
        
        # Attention is applied across the batch of symbols, not across time
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_dim, num_heads=num_heads, 
            dropout=dropout, batch_first=True
        )
        self.norm = nn.LayerNorm(hidden_dim)
        
        self.output_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_targets)
        )

    def forward(self, x: torch.Tensor, padding_mask: torch.Tensor = None) -> torch.Tensor:
        h = self.input_proj(x)
        attn_out, _ = self.attention(
            query=h, key=h, value=h, key_padding_mask=padding_mask
        )
        h = self.norm(h + attn_out)
        return self.output_head(h)

## V. The Inference Challenge: PyTorch Statefulness & Memory Leaks

The true engineering challenge of this competition was executing the ensemble within the live API.

Because the API feeds data one row at a time (seq_len=1), the GRU naturally loses its hidden state ($h_t$) after every millisecond. The model thinks every single row is the first minute of a brand new day. To fix this, the hidden state had to be manually intercepted, stored in a dictionary mapped to the specific symbol_id, and injected back into the model on the next loop.

The "Silent Killers" (OOM Crashes)

Executing a Python for loop 327,000+ times exposed two massive memory leaks:

1. CPU Bloat: Polars/Pandas dataframes are immutable. Running .filter() to extract the current batch instantiated a new DataFrame object 327,000 times, overwhelming the Python garbage collector and crashing the system RAM.
2. GPU VRAM Leak: When saving the GRU state (h_out[:, idx, :]), PyTorch tensor slicing secretly maintains the computational graph of the entire batch. Doing this in a loop flooded the GPU.

The Solution: A zero-allocation, NumPy-accelerated streaming loop. By pre-extracting the data to contiguous C-arrays, finding step boundaries mathematically, and explicitly using .detach().clone() on the PyTorch tensors, the inference pipeline bypassed the memory bottlenecks entirely.

In [ ]:
# 1. Pre-extract all data to contiguous NumPy arrays to kill DataFrame reallocation overhead
all_gru_static = val_df.select(gru_static_cols).to_numpy().astype(np.float32)
all_gru_tv     = val_df.select(gru_tv_cols).to_numpy().astype(np.float32)
all_keys       = val_df.select(["date_id", "time_id", "symbol_id"]).to_numpy().astype(np.int32)

# 2. Find step boundaries mathematically using NumPy vectorization
date_time = all_keys[:, :2]
changes = np.any(date_time[1:] != date_time[:-1], axis=1)
boundaries = np.nonzero(changes)[0] + 1
splits = np.split(np.arange(len(val_df)), boundaries)

# 3. Initialize O(1) memory buffers mapped perfectly to individual symbols
unique_dates = np.unique(all_keys[:, 0])
gru_memory = {d_id: {sym: None for sym in range(39)} for d_id in unique_dates}

with torch.no_grad():
    for i, indices in enumerate(splits):
        start, end = indices[0], indices[-1] + 1
        d_id = all_keys[start, 0]
        current_symbols = all_keys[start:end, 2].tolist()
        
        # --- Stateful GRU Inference ---
        g_static = torch.tensor(all_gru_static[start:end], device=device)
        g_tv     = torch.tensor(np.expand_dims(all_gru_tv[start:end], axis=1), device=device)
        
        # Fetch existing memories, replacing missing symbols with zero-tensors dynamically
        h_in_list = [gru_memory[d_id][sym] for sym in current_symbols]
        if all(h is None for h in h_in_list):
            h_in_tensor = None
        else:
            valid_h = next(h for h in h_in_list if h is not None)
            clean_h = [h if h is not None else torch.zeros_like(valid_h) for h in h_in_list]
            h_in_tensor = torch.stack(clean_h, dim=1).to(device)
        
        # Forward pass: Feed in features + matched memory, catch the new memory
        gru_out, h_out = gru_model(g_static, g_tv, h_in=h_in_tensor)
        
        h_out_detached = h_out.detach().clone() 
        for idx, sym in enumerate(current_symbols):
            gru_memory[d_id][sym] = h_out_detached[:, idx, :]
            
        pred_gru = gru_out[:, :, 0].cpu().numpy().flatten()

## VI. Online Fine-Tuning

The ultimate defense against non-stationary financial data (regime shifts) is allowing the model to learn during inference.

In this competition, the streaming API occasionally returns a lags dataframe containing the true targets of previous days. I engineered an online_finetune_step that intercepts these revealed targets. By freezing the deep static encoders of the GRU (to prevent catastrophic forgetting) and unfreezing only the final output head, the pipeline executes a micro-backward pass on the new data. This allows the ensemble's weights to dynamically drift and adapt to the current market regime before making predictions on the present day.

## VII. Final Results & Conclusion

By engineering cross-sectional features to capture market-wide momentum, mathematically reconstructing hidden states in a streaming environment, and bypassing catastrophic Python memory leaks, the final pipeline proved exceptionally resilient.

Here is the progression of the out-of-sample validation scoring (Partitions 8-9):

| Model Architecture | Technique Highlight | Validation $R^2$ |
| --- | --- | --- |
| **LightGBM (Baseline)** | Standard Regression | -0.4479 |
| **Jane Street GRU (Stateless)** | `seq_len=1` (Amnesia) | 0.8487 |
| **Jane Street GRU (Stateful)** | $\mathcal{O}(1)$ Continuous Memory Tracking | 0.8521 |
| **Final Ensemble** | CST (88%) / GRU (12%) Blend | **0.8650** |

The ensemble successfully acts as a stabilizing mechanism. The Cross-Symbol Transformer anchors the predictions based on instantaneous market weather, while the stateful GRU provides deep historical context, culminating in a robust, low-latency prediction engine capable of navigating severe market turbulence.